CRSP - Market Reaction around sustainability reporting release date

* 8/24/2026

In [1]:
# Import the required packages

import pandas as pd
import numpy as np
import wrds
import warnings

In [2]:
conn=wrds.Connection()

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [6]:
###################
# CRSP Block      #
###################
# sql similar to crspmerge macro
# prc: Price or Bid/Ask Average 

crsp = conn.raw_sql("""
                      select a.permno, a.permco, a.date, b.ticker, b.shrcd, b.exchcd, a.cusip,
                      a.ret, a.retx, a.shrout, a.prc
                      from crsp.dsf as a
                      left join crsp.dsenames as b                     
                      on a.permno=b.permno
                      and b.namedt<=a.date
                      and a.date<=b.nameendt
                      left join crsp.dsedelist as c 
                      on a.permno = c.permno  
                      where a.date between '12/1/2017' and '01/31/2024'
                      """, coerce_float=True, date_cols=None)


vwretd = conn.raw_sql("""
    SELECT
        caldt AS date,
        vwretd
    FROM crsp.dsix
    WHERE caldt BETWEEN '2017-12-01' AND '2024-01-31'
""", coerce_float=True, date_cols=['date'])

In [8]:
crsp['date'] = pd.to_datetime(crsp['date'])
vwretd['date'] = pd.to_datetime(vwretd['date'])


crsp = crsp.merge(
    vwretd,
    on='date',
    how='left',
    validate='many_to_one'
)

In [10]:
sasb = pd.read_excel(r"D:\Dropbox\Dropbox\Dropbox\4_SASB\Data and Coding\SASB Metrics and Data (working).xlsx", sheet_name = 'Sample')
sasb.columns = sasb.columns.str.strip().str.lower()

sasb = sasb[['line in master', 'company name', 'press release date']].copy()

sasb.rename(
    columns={'press release date': 'date_release'},
    inplace=True
)

# Make sure release date is datetime
sasb['date_release'] = pd.to_datetime(
    sasb['date_release'],
    errors='coerce'
)

# One year before release
sasb['date_1yrbrelease'] = (
    sasb['date_release'] - pd.DateOffset(years=1)
)

sasb.head()

,line in master,company name,date_release,date_1yrbrelease
0,10,AdvanSix Inc,2021-05-10,2020-05-10
1,11,Aflac Inc,2020-03-04,2019-03-04
2,12,Agilent Technologies Inc,2021-07-22,2020-07-22
3,14,AGNC Investment Corp,2021-03-15,2020-03-15
4,15,AIG - American International Group Inc,2021-06-30,2020-06-30


In [11]:
identifier = pd.read_excel(r"D:\Dropbox\Dropbox\Dropbox\4_SASB\Data and Coding\SASB Metrics and Data (working).xlsx", sheet_name = 'Compustat Links')
identifier.columns = identifier.columns.str.strip().str.lower()
identifier.rename(
    columns={'sasb sample name': 'company name'},
    inplace=True
)
identifier = identifier[['company name', 'cusip','gvkey', 'conm', 'tic', 'cik']].copy()
sasb = pd.merge(sasb, identifier, on='company name', how='left', indicator=True)

sasb.drop(columns=['_merge'], inplace=True)
sasb.head()

,line in master,company name,date_release,date_1yrbrelease,cusip,gvkey,conm,tic,cik
0,10,AdvanSix Inc,2021-05-10,2020-05-10,00773T101,28070,ADVANSIX INC,ASIX,1673985
1,11,Aflac Inc,2020-03-04,2019-03-04,1055102,1449,AFLAC INC,AFL,4977
2,12,Agilent Technologies Inc,2021-07-22,2020-07-22,00846U101,126554,AGILENT TECHNOLOGIES INC,A,1090872
3,14,AGNC Investment Corp,2021-03-15,2020-03-15,00123Q104,179889,AGNC INVESTMENT CORP,AGNC,1423689
4,15,AIG - American International Group Inc,2021-06-30,2020-06-30,26874784,1487,AMERICAN INTERNATIONAL GROUP,AIG,5272


In [12]:
sasb['cusip9'] = (
    sasb['cusip']
    .astype(str)
    .str.strip()
    .str.zfill(9)
)

sasb['cusip8'] = (
    sasb['cusip9']
    .astype(str)
    .str.strip()
    .str.upper()
    .str[:8]
)

sasb.head()

,line in master,company name,date_release,date_1yrbrelease,cusip,gvkey,conm,tic,cik,cusip9,cusip8
0,10,AdvanSix Inc,2021-05-10,2020-05-10,00773T101,28070,ADVANSIX INC,ASIX,1673985,00773T101,00773T10
1,11,Aflac Inc,2020-03-04,2019-03-04,1055102,1449,AFLAC INC,AFL,4977,001055102,00105510
2,12,Agilent Technologies Inc,2021-07-22,2020-07-22,00846U101,126554,AGILENT TECHNOLOGIES INC,A,1090872,00846U101,00846U10
3,14,AGNC Investment Corp,2021-03-15,2020-03-15,00123Q104,179889,AGNC INVESTMENT CORP,AGNC,1423689,00123Q104,00123Q10
4,15,AIG - American International Group Inc,2021-06-30,2020-06-30,26874784,1487,AMERICAN INTERNATIONAL GROUP,AIG,5272,026874784,02687478


In [13]:
crsp.head()

,permno,permco,date,ticker,shrcd,exchcd,cusip,ret,retx,shrout,prc,vwretd
0,10026,7976,2017-12-01,JJSF,11,3,46603210,-0.006882,-0.006882,18664.0,150.07001,-0.001075
1,10028,7978,2017-12-01,DGSE,11,2,29402E10,0.009615,0.009615,26924.0,1.05000,-0.001075
2,10032,7980,2017-12-01,PLXS,11,3,72913210,-0.011358,-0.011358,33585.0,61.80000,-0.001075
3,10044,7992,2017-12-01,RMCF,11,3,77467X10,0.023478,0.023478,5903.0,11.77000,-0.001075
4,10051,7999,2017-12-01,None,11,0,41043F20,NaN,NaN,35291.0,NaN,-0.001075


In [14]:
crsp.rename(columns={'cusip': 'cusip8'}, inplace=True)

In [15]:
merge = pd.merge(sasb, crsp, on=['cusip8'], how='left', indicator=True)
merge['_merge'].value_counts()

_merge
both          454631
left_only          1
right_only         0
Name: count, dtype: int64

In [16]:
merge[merge['_merge'] == 'left_only'][['company name', 'cusip8', 'date_release', 'date_1yrbrelease', 'tic']].head(10)

## Kinder Morgan Canada Ltd is missing from CRSP. It is a Canadian company and not listed on a US exchange.

,company name,cusip8,date_release,date_1yrbrelease,tic
229189,Kinder Morgan Canada Ltd,49454970,2019-10-24,2018-10-24,KMLGF


In [22]:
merge.columns

Index(['line in master', 'company name', 'date_release', 'date_1yrbrelease',
       'cusip', 'gvkey', 'conm', 'tic', 'cik', 'cusip9', 'cusip8', 'permno',
       'permco', 'date', 'ticker', 'shrcd', 'exchcd', 'ret', 'retx', 'shrout',
       'prc', 'vwretd', '_merge'],
      dtype='object')

In [23]:
## Filter dataset -1 to +1 days around the release date
df = merge[['line in master', 'date_release', 'date', 'ret', 'prc', 'shrout', 'vwretd']]
df.head()

,line in master,date_release,date,ret,prc,shrout,vwretd
0,10,2021-05-10,2017-12-01,-0.046690,41.04,30483.0,-0.001075
1,10,2021-05-10,2017-12-04,-0.026316,39.96,30483.0,-0.001809
2,10,2021-05-10,2017-12-05,0.008258,40.29,30483.0,-0.004225
3,10,2021-05-10,2017-12-06,-0.008439,39.95,30483.0,-0.001315
4,10,2021-05-10,2017-12-07,0.009762,40.34,30483.0,0.004137


CAR[-1,+1]

In [25]:
# Make sure dates are datetime
df = df.copy()
df['date'] = pd.to_datetime(df['date'])
df['date_release'] = pd.to_datetime(df['date_release'])

# Sort by report and trading date
df = df.sort_values(['line in master', 'date'])

# Create trading-day sequence within each report
df['trade_num'] = df.groupby('line in master').cumcount()

# Find first trading day ON or AFTER release date
event_trade_num = (
    df[df['date'] >= df['date_release']]
    .groupby('line in master')['trade_num']
    .min()
    .rename('event_trade_num')
)

# Merge back
df = df.merge(
    event_trade_num,
    on='line in master',
    how='left'
)

# Trading-day event time
df['event_time'] = df['trade_num'] - df['event_trade_num']

df['abret'] = df['ret'] - df['vwretd']

In [26]:
df.columns

Index(['line in master', 'date_release', 'date', 'ret', 'prc', 'shrout',
       'vwretd', 'trade_num', 'event_trade_num', 'event_time', 'abret'],
      dtype='object')

In [27]:
event_window = df[df['event_time'].between(-1, 1)].copy()
df

car3 = (
    event_window
    .groupby('line in master')['abret']
    .apply(lambda x: (1 + x).prod() - 1)
    .reset_index(name='car3')
)

In [28]:
car3.to_stata("car3.dta")

C:\Users\huipi\AppData\Local\Temp\ipykernel_2600\2757603790.py:1: InvalidColumnName: 
Not all pandas column names were valid Stata variable names.
The following replacements have been made:

    line in master   ->   line_in_master

If this is not what you expect, please make sure you have Stata-compliant
column names in your DataFrame (strings only, max 32 characters, only
alphanumerics and underscores, no Stata reserved words)

  car3.to_stata("car3.dta")
